### Metadata
**Name**  
> Naveed Ahmad Hematmal

**Assignment**    

> Matrix Decomposition

# Mathematics for Machine Learning

# Laboratory 4b: Matrix Decomposition using PyTorch

---

## Topics

1. Eigen Decomposition and Diagonalization
2. Singular Value Decomposition
3. Matrix Approximation

## Environment

- PyTorch
- Matplotlib
- Jupyter Notebook or Google Colab

This lab uses small, reproducible examples and does not require external data.

---

## Learning Objectives

By the end of this laboratory, you should be able to:

- Compute and interpret eigenvalues and eigenvectors of square matrices.
- Diagonalize a matrix when a complete eigenvector basis exists, and recognize a defective matrix that cannot be diagonalized.
- Compute a compact singular value decomposition (SVD) of a rectangular matrix.
- Construct a rank-$r$ approximation and evaluate its error.
- Relate low-rank approximation to compression and dimensionality reduction.

Throughout the notebook, use `torch.float64` for the worked examples so that numerical residuals are easy to inspect. Small floating-point differences are expected; use tolerances rather than exact equality for numerical identities.

## Setup

The only imports are PyTorch and Matplotlib. The fixed random seed makes the synthetic examples reproducible.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)
DTYPE = torch.float64

# 1. Eigen Decomposition and Diagonalization

For a square matrix $A$, an eigenpair $(\lambda, \mathbf{v})$ satisfies

$$A\mathbf{v} = \lambda\mathbf{v}, \qquad \mathbf{v} \ne \mathbf{0}.$$

If $A$ has a basis of linearly independent eigenvectors, place them in the columns of $P$ and their eigenvalues on the diagonal of $D$. Then

$$A = PDP^{-1}, \qquad A^k = PD^kP^{-1}.$$

A real symmetric matrix has real eigenvalues and an orthonormal eigenbasis. For such matrices, `torch.linalg.eigh` is preferred over the general `eig` routine.

## 1.1 Eigenpairs of a Symmetric Matrix

We start with a symmetric matrix whose eigenvalues are distinct. The `eigh` routine returns eigenvalues in ascending order; its eigenvectors are the columns of `Q`.

In [ ]:
A = torch.tensor([[4.0, 1.0, 1.0],
                  [1.0, 3.0, 0.0],
                  [1.0, 0.0, 2.0]], dtype=DTYPE)
eigenvalues, Q = torch.linalg.eigh(A)

print("A =\n", A)
print("eigenvalues =", eigenvalues)
print("eigenvectors (columns of Q) =\n", Q)

Each column $\mathbf{q}_i$ of `Q` should satisfy $A\mathbf{q}_i = \lambda_i\mathbf{q}_i$. Because this matrix is symmetric, `Q` should also be orthogonal: $Q^TQ = I$.

In [ ]:
I = torch.eye(A.shape[0], dtype=DTYPE)
eigenpair_residual = torch.linalg.norm(A @ Q - Q @ torch.diag(eigenvalues))
orthogonality_residual = torch.linalg.norm(Q.T @ Q - I)
reconstruction_residual = torch.linalg.norm(A - Q @ torch.diag(eigenvalues) @ Q.T)

print(f"Eigenpair residual:      {eigenpair_residual:.3e}")
print(f"Orthogonality residual:  {orthogonality_residual:.3e}")
print(f"Reconstruction residual: {reconstruction_residual:.3e}")
assert torch.allclose(A @ Q, Q @ torch.diag(eigenvalues), atol=1e-10)
assert torch.allclose(Q.T @ Q, I, atol=1e-10)

## 1.2 Diagonalization of a General Matrix

Symmetry is sufficient for diagonalizability, but it is not necessary. A matrix with $n$ linearly independent eigenvectors can be diagonalized even when it is not symmetric. Consider the upper-triangular matrix below. Its distinct diagonal entries are its eigenvalues.

In [ ]:
B = torch.tensor([[2.0, 1.0, 0.0],
                  [0.0, 3.0, 1.0],
                  [0.0, 0.0, 5.0]], dtype=DTYPE)
values, vectors = torch.linalg.eig(B)

# This example has real eigenvalues/eigenvectors. Keep real tensors for the checks.
values = values.real
P = vectors.real
D = torch.diag(values)
P_inverse = torch.linalg.inv(P)

print("eigenvalues =", values)
print("condition number of P =", torch.linalg.cond(P).item())
print("||B - P D P^{-1}||_F =", torch.linalg.norm(B - P @ D @ P_inverse).item())
assert torch.allclose(B, P @ D @ P_inverse, atol=1e-10)

The factorization allows powers of a matrix to be computed through the diagonal matrix: $B^k = PD^kP^{-1}$. This is useful in analyzing repeated linear transformations and discrete-time dynamical systems. If eigenvalues have magnitude greater than one, components in their eigendirections grow under repeated application; magnitudes below one lead to decay.

In [ ]:
k = 5
B_power_direct = torch.linalg.matrix_power(B, k)
B_power_diagonal = P @ torch.linalg.matrix_power(D, k) @ P_inverse
print("B^5 computed directly =\n", B_power_direct)
print(f"Diagonalization identity residual: {torch.linalg.norm(B_power_direct - B_power_diagonal):.3e}")

## 1.3 When Diagonalization Fails

Not every square matrix has a complete eigenvector basis. Consider the Jordan block

$$J = \begin{bmatrix}2 & 1 \\ 0 & 2\end{bmatrix}.$$

Its characteristic polynomial has a repeated root, but its eigenspace is only one-dimensional. Therefore no invertible $P$ can satisfy $J = PDP^{-1}$.

In [ ]:
J = torch.tensor([[2.0, 1.0],
                  [0.0, 2.0]], dtype=DTYPE)
shifted = J - 2 * torch.eye(2, dtype=DTYPE)
print("rank(J - 2I) =", torch.linalg.matrix_rank(shifted).item())
print("nullity(J - 2I) =", (2 - torch.linalg.matrix_rank(shifted)).item())
print("algebraic multiplicity of eigenvalue 2 = 2")
print("geometric multiplicity of eigenvalue 2 = nullity(J - 2I) = 1")

### Check Your Understanding

1. Why does the symmetric matrix in Section 1.1 have an orthonormal eigenbasis?
2. For the matrix $B$, compute the eigenvalues by hand from its triangular form and compare them with PyTorch's result.
3. For the Jordan block $J$, verify that $(J - 2I)\mathbf{v} = 0$ has only one independent solution direction. What condition on eigenvectors makes diagonalization possible?

**Instructor check:** diagonalizability requires $n$ linearly independent eigenvectors for an $n \times n$ matrix. Repeated eigenvalues are allowed, but their eigenspaces must provide enough independent eigenvectors.

# 2. Singular Value Decomposition

Every real matrix $X \in \mathbb{R}^{m\times n}$ has an SVD

$$X = U\Sigma V^T,$$

where the columns of $U$ and $V$ are orthonormal and the singular values on the diagonal of $\Sigma$ are nonnegative and conventionally sorted from largest to smallest. Unlike eigen decomposition, SVD applies to rectangular matrices as well as square ones.

For a compact SVD, `torch.linalg.svd(X, full_matrices=False)` returns shapes

- `U`: $(m, r)$,
- `S`: $(r,)$,
- `Vh`: $(r, n)$,

where $r=\min(m,n)$ and `Vh` is $V^T$. The singular values measure how strongly the matrix stretches vectors in corresponding orthogonal directions.

## 2.1 Compute and Verify an SVD

We construct a rectangular matrix with a clear low-rank component and a small perturbation. First check reconstruction and orthogonality.

In [ ]:
torch.manual_seed(12)
left_factor = torch.randn(6, 2, dtype=DTYPE)
right_factor = torch.randn(2, 4, dtype=DTYPE)
X = left_factor @ right_factor + 0.05 * torch.randn(6, 4, dtype=DTYPE)
U, S, Vh = torch.linalg.svd(X, full_matrices=False)
X_reconstructed = U @ torch.diag(S) @ Vh

print("X shape:", tuple(X.shape))
print("U, S, Vh shapes:", tuple(U.shape), tuple(S.shape), tuple(Vh.shape))
print("singular values:", S)
print(f"Reconstruction error: {torch.linalg.norm(X - X_reconstructed):.3e}")
print(f"U orthogonality error: {torch.linalg.norm(U.T @ U - torch.eye(U.shape[1], dtype=DTYPE)):.3e}")
print(f"V orthogonality error: {torch.linalg.norm(Vh @ Vh.T - torch.eye(Vh.shape[0], dtype=DTYPE)):.3e}")

For a matrix $X$, the squared singular values are the eigenvalues of $X^TX$ (and also of $XX^T$, up to extra zero eigenvalues when dimensions differ). This provides a useful bridge between eigen decomposition and SVD.

In [ ]:
right_gram_eigenvalues = torch.linalg.eigvalsh(X.T @ X).clamp_min(0).flip(0)
print("S^2 =", S.square())
print("eigenvalues of X^T X =", right_gram_eigenvalues)
assert torch.allclose(S.square(), right_gram_eigenvalues, atol=1e-10)

# 3. Matrix Approximation

Truncating an SVD after the largest $k$ singular values gives

$$X_k = U_{[:,1:k]}\,\Sigma_{[1:k,1:k]}\,V^T_{[1:k,:]}.$$

The Eckart--Young theorem states that $X_k$ is a best rank-$k$ approximation to $X$ in both the Frobenius norm and the spectral norm. The errors are determined by the discarded singular values:

$$\|X-X_k\|_F = \sqrt{\sum_{i>k}\sigma_i^2}, \qquad
\|X-X_k\|_2 = \sigma_{k+1}.$$

Here the Frobenius norm measures total squared entrywise error, while the spectral norm measures the largest singular value of the error matrix.

## 3.1 Approximation Error versus Rank

We compare several truncations of the same matrix. The printed errors should agree with the tail singular-value formulas, up to numerical precision.

In [ ]:
def rank_k_approximation(U, S, Vh, k):
    """Return the rank-k truncated SVD reconstruction."""
    if not 1 <= k <= S.numel():
        raise ValueError(f"k must be between 1 and {S.numel()}, got {k}")
    return (U[:, :k] * S[:k]) @ Vh[:k, :]

print(" rank | Frobenius error | tail formula | spectral error | tail sigma")
for rank in (1, 2, 3):
    X_rank = rank_k_approximation(U, S, Vh, rank)
    error = X - X_rank
    frobenius_error = torch.linalg.norm(error, ord="fro")
    frobenius_tail = torch.linalg.norm(S[rank:])
    spectral_error = torch.linalg.norm(error, ord=2)
    spectral_tail = S[rank] if rank < S.numel() else torch.tensor(0.0, dtype=DTYPE)
    print(f" {rank:4d} | {frobenius_error:15.6f} | {frobenius_tail:11.6f} | {spectral_error:14.6f} | {spectral_tail:10.6f}")

## 3.2 Singular-Value Energy

A common heuristic is to choose the smallest rank that retains a desired fraction of the squared singular-value energy. This is useful for compression, but the threshold should be chosen based on the task and acceptable error rather than treated as a universal rule.

In [ ]:
energy = S.square()
energy_fraction = energy.cumsum(0) / energy.sum()
for rank, fraction in enumerate(energy_fraction, start=1):
    print(f"rank {rank}: retained energy = {fraction.item():.2%}")

## 3.3 Low-Rank Image Approximation

An image can be represented as a matrix of pixel intensities. We create a small synthetic grayscale image, compute its SVD, and display several low-rank reconstructions. The example is self-contained and does not download an image.

In [ ]:
side = 64
y = torch.linspace(0, 1, side, dtype=DTYPE)
x = torch.linspace(0, 1, side, dtype=DTYPE)
Y, X_grid = torch.meshgrid(y, x, indexing="ij")
image_clean = (
    0.50
    + 0.20 * torch.sin(2 * torch.pi * X_grid)
    + 0.15 * torch.cos(2 * torch.pi * Y)
    + 0.10 * (X_grid * Y)
)
torch.manual_seed(21)
image = (image_clean + 0.06 * torch.randn_like(image_clean)).clamp(0, 1)
image_U, image_S, image_Vh = torch.linalg.svd(image, full_matrices=False)

ranks = (1, 5, 15, 40)
reconstructions = [rank_k_approximation(image_U, image_S, image_Vh, rank) for rank in ranks]
print("Image matrix shape:", tuple(image.shape))
print("First ten singular values:", image_S[:10])

In [ ]:
fig, axes = plt.subplots(1, len(ranks) + 1, figsize=(14, 3.2))
axes[0].imshow(image, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Noisy image")
axes[0].axis("off")
for axis, rank, reconstruction in zip(axes[1:], ranks, reconstructions):
    axis.imshow(reconstruction, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"Rank {rank}")
    axis.axis("off")
plt.tight_layout()
plt.show()

The reconstructions become more detailed as rank increases. The first few singular directions often capture large-scale structure, while later directions restore finer detail and noise. In real image compression, storage cost and visual quality must both be considered; the SVD itself is not necessarily the most efficient encoding for every image.

In [ ]:
image_energy = image_S.square()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(torch.arange(1, image_S.numel() + 1), image_S, marker=".", linewidth=1)
ax.set_xlabel("Singular value index")
ax.set_ylabel("Singular value (log scale)")
ax.set_title("Image singular-value spectrum")
ax.grid(True, which="both", alpha=0.3)
plt.show()

for rank, reconstruction in zip(ranks, reconstructions):
    relative_error = torch.linalg.norm(image - reconstruction) / torch.linalg.norm(image)
    retained = image_energy[:rank].sum() / image_energy.sum()
    print(f"rank {rank:2d}: relative Frobenius error = {relative_error:.3%}; retained energy = {retained:.3%}")

## Summary

- Eigen decomposition represents a square matrix in an eigenvector basis when enough independent eigenvectors exist.
- Real symmetric matrices have real eigenvalues and orthonormal eigenvectors; use `torch.linalg.eigh` for this case.
- SVD applies to any real matrix and separates its action into orthogonal directions and nonnegative singular values.
- Truncated SVD gives the best rank-$k$ approximation in Frobenius and spectral norms.
- Singular values quantify the approximation error and provide a practical guide for compression and dimensionality reduction.



## Exercises

1. Change the symmetric matrix in Section 1.1. Which properties remain true if you use any real symmetric matrix? Which properties are specific to having distinct eigenvalues?
2. Construct a diagonalizable matrix with a repeated eigenvalue. Confirm that the eigenvectors returned by PyTorch span the full space.
<!-- 3. For the Jordan block, compute $J^k$ for several values of $k$. Explain why the result cannot be obtained by diagonalizing $J$. -->
3. Create a rectangular matrix with a known rank, for example $X=FG$ with a small inner dimension. How many singular values should be nonzero in exact arithmetic?
4. For the image example, choose the smallest rank that retains at least 95% of squared singular-value energy. Plot that reconstruction and report its relative Frobenius error.
5. Why can a low-rank approximation reduce storage? Compare storing $m n$ matrix entries with storing $U_{[:,1:k]}$, the first $k$ singular values, and $V^T_{[1:k,:]}$.



---

**End of Laboratory 4**